<a href="https://colab.research.google.com/github/zorGizem/Erken-Evre-Alzhemir-Tespiti/blob/main/notebooks/07_normalizsayon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neden Yoğunluk Normalizasyonu (Intensity Normalization) Yapıldı?

MRI görüntüleri, çekim yapılan cihazın parametrelerine ve manyetik alan gücüne bağlı olarak farklı yoğunluk aralıklarına sahip olabilir. Bu durum, aynı doku tipinin farklı görüntülerde çok farklı sayısal değerlerle temsil edilmesine ve modelin öğrenme sürecinin bozulmasına yol açar. Bu sorunu gidermek için uygulanan Z-skoru normalizasyonu, her bir görüntünün piksel değerlerini kendi ortalamasından çıkarıp standart sapmasına bölerek veriyi sıfır ortalama ve bir standart sapma etrafında toplar. Bu sayede tüm hastaların beyin görüntüleri sayısal olarak karşılaştırılabilir hale getirilmiş, modelin daha hızlı yakınsaması ve öznitelikleri daha kararlı bir şekilde öğrenmesi sağlanmıştır.

In [ ]:
import numpy as np
import nibabel as nib
print(" Kütüphaneler hazır")

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU Aktif: {torch.cuda.get_device_name(0)}")
else:
    print("GPU bulunamadı!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
CONFIG_CN = {
    "kategori"    : "CN",
    "kaynak": f'/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/Registrasionn/Registrasion_CN',
    "hedef" : f'/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/Normalizasyonn/Normalizasyon_CN',
}

CONFIG_EMCI = {
    "kategori"    : "EMCI",
    "kaynak": f'/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/Registrasionn/Registrasion_EMCI',
    "hedef" : f'/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/Normalizasyonn/Normalizasyon_EMCI',
}

In [ ]:

import os
import shutil
import numpy as np
import nibabel as nib

def batch_zscore_normalizasyon(config, batch_size=20):
    source_base = config["kaynak"]
    output_base = config["hedef"]
    etiket = config["kategori"]

    local_in = "/content/batch_in_norm"
    local_out = "/content/batch_out_norm"

    print(f"\n {etiket} Z-SKOR NORMALİZASYON Başlıyor (Grup Boyutu: {batch_size})")


    all_pending = []
    subjects = sorted([s for s in os.listdir(source_base) if os.path.isdir(os.path.join(source_base, s))])

    toplam_dosya = 0
    atlanan_dosya = 0
    basarili_dosya = 0
    hatali_dosya = 0

    print(" Klasörler taranıyor, lütfen bekleyin...", end="\r")

    for subject_id in subjects:
        subject_path = os.path.join(source_base, subject_id)
        seanslar = [d for d in os.listdir(subject_path) if os.path.isdir(os.path.join(subject_path, d))]

        for seans in seanslar:
            seans_yolu = os.path.join(subject_path, seans)
            nifti_files = [f for f in os.listdir(seans_yolu) if f.endswith('.nii.gz')]

            if nifti_files:
                toplam_dosya += 1
                f = nifti_files[0]
                rel_path = os.path.join(subject_id, seans)
                cikti_yolu = os.path.join(output_base, rel_path, f)

                if not os.path.exists(cikti_yolu):
                    all_pending.append((os.path.join(seans_yolu, f), rel_path, f, subject_id))
                else:
                    atlanan_dosya += 1

    # --- ÖN ANALİZ RAPORU ---
    kalan_dosya = len(all_pending)
    print(" "*50, end="\r") # Tarama yazısını temizle
    print(f" KLASÖR ANALİZİ ({etiket}):")
    print(f"    Toplam Bulunan Dosya : {toplam_dosya}")
    print(f"    Zaten İşlenmiş Olan  : {atlanan_dosya}")
    print(f"    Şimdi İşlenecek Olan : {kalan_dosya}")
    print("-" * 40)

    if not all_pending:
        print(f" İşlenecek yeni dosya yok. Sistem durduruluyor.")
    else:
        # 2. Döngü
        for i in range(0, kalan_dosya, batch_size):
            batch = all_pending[i:i+batch_size]
            shutil.rmtree(local_in, ignore_errors=True)
            shutil.rmtree(local_out, ignore_errors=True)
            os.makedirs(local_in, exist_ok=True)
            os.makedirs(local_out, exist_ok=True)

            batch_subjects = sorted(list(set([item[3] for item in batch])))
            print(f"\n Grup {i//batch_size + 1} Başlıyor...")
            print(f" İşlenen Hastalar: {', '.join(batch_subjects)}")

            mapping = {}
            for full_src, rel_p, fname, sid in batch:
                unique_name = rel_p.replace("/", "_") + "___" + fname
                local_girdi = os.path.join(local_in, unique_name)
                shutil.copy2(full_src, local_girdi)
                mapping[unique_name] = (rel_p, fname, local_girdi)

            print(f" Normalizasyon (Z-Skor) çalışıyor... ", end="", flush=True)

            try:
                for u_name, (rel_p, fname, l_girdi) in mapping.items():
                    l_out_file = os.path.join(local_out, u_name)


                    img = nib.load(l_girdi)
                    veri = img.get_fdata()


                    maske = veri > 0

                    if np.any(maske): # Eğer tamamen siyah bir resim değilse
                        beyin_vokselleri = veri[maske]
                        mean = beyin_vokselleri.mean()
                        std = beyin_vokselleri.std()

                        # Sadece maske içindeki alanlara Z-Score uygula
                        normalize_veri = np.zeros_like(veri)
                        if std > 0: # Sıfıra bölme hatasını önlemek için güvenlik
                            normalize_veri[maske] = (veri[maske] - mean) / std
                    else:
                        normalize_veri = veri # Hatalı/boş dosya ise aynen bırak

                    yeni_img = nib.Nifti1Image(normalize_veri, img.affine, img.header)
                    nib.save(yeni_img, l_out_file)
                    # ========================================================

                print(" Bitti")

                print(f" Drive'a aktarılıyor... ", end="", flush=True)
                for u_name, (rel_p, fname, _) in mapping.items():
                    l_res = os.path.join(local_out, u_name)
                    d_target_dir = os.path.join(output_base, rel_p)
                    os.makedirs(d_target_dir, exist_ok=True)

                    if os.path.exists(l_res):
                        shutil.move(l_res, os.path.join(d_target_dir, fname))
                        basarili_dosya += 1

                su_an_biten = min(i+batch_size, kalan_dosya)
                print(f" {su_an_biten}/{kalan_dosya} yeni dosya kaydedildi.")

            except Exception as e:
                print(f" Grup Hatası: {str(e)}")
                hatali_dosya += len(batch)

    # STANDART ÖZET YAPISI
    print(f"\n{etiket} Z-Skor Normalizasyon Özeti:")
    print(f"Toplam   : {toplam_dosya}")
    print(f"Başarılı : {basarili_dosya}")
    print(f"Atlanan  : {atlanan_dosya}")
    print(f"Hatalı   : {hatali_dosya}")
    print(f"{etiket} grubu tamamlandı.")

In [ ]:
batch_zscore_normalizasyon(CONFIG_CN)

In [ ]:
batch_zscore_normalizasyon(CONFIG_EMCI)